In [2]:
# =============================================================================
# Korea Trade Data Collector (V6)
# V6 변경 (2026-10-01)
# - HS 코드 목록: hs_code_map_v2 (누락 코드 6개 추가: 030341~030344, 0207, 190490)
# - 서비스키/DB 비밀번호를 코드에서 제거 → DATA/KEYS.py, DATA/config.py 에서 읽음
# - TARGET_CODES: 특정 코드만 골라 과거부터 재수집 가능 (누락 코드 백필용)
# - YoY 계산용으로 시작월 12개월 전부터 함께 받음 (V5는 1년치만 받아 YoY가 비어 있었음)
# - DB 저장: 기존 키는 건너뛰던 방식 → 값 갱신(UPSERT). 수정치 반영, 전체 키 조회 제거로 빨라짐
# - AUTO_UPLOAD: 업로드 여부를 설정으로 지정 가능 (None 이면 기존처럼 물어봄)
# - HS별 1회 작업(내부에서 12개월 window로 자동 분할)
# - YYYYMM 범위 수집 (START_YYYYMM ~ END_YYYYMM)
# - API 제약: 조회기간 1년(12개월) 이내 → 자동 분할 후 concat
# - 안전 파싱 / 재시도 / 세션 풀 / 6자리 HS 요청 대응
# - 월/분기 집계 + YoY + Long format + DB 업로드
# =============================================================================

import re
import time
import logging
import warnings

import requests
import xmltodict
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd
from tqdm import tqdm

from sqlalchemy import create_engine, text
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter

# -----------------------------------------------------------------------------
# 경로 설정 (노트북 / 데스크탑 공용)
#   1순위: 현재 위치에서 상위 폴더로 올라가며 DATA 폴더 자동 탐색
#   2순위: 두 PC의 알려진 고정 경로 fallback
#   → "from DATA.xxx import ..." 가 작동하려면 DATA의 "상위 폴더(프로젝트 루트)"가
#     sys.path에 있어야 하므로, 루트와 DATA 폴더를 모두 추가한다.
# -----------------------------------------------------------------------------
import sys
import os
from pathlib import Path

# 두 PC의 알려진 DATA 폴더 경로 (fallback용)
KNOWN_DATA_PATHS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA",          # 데스크탑
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA",  # 노트북
]

def setup_universal_paths():
    """어떤 PC에서도 작동하는 범용 경로 설정: DATA 폴더 자동 탐색 + 고정 경로 fallback"""
    # 1) 현재 위치 기준 상위 폴더 자동 탐색
    current = Path.cwd()
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            for p in (str(parent), str(data_folder)):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print(f"[경로 설정] 자동 탐색 성공 | 프로젝트 루트: {parent}")
            print(f"[경로 설정] DATA 폴더: {data_folder}")
            return str(data_folder)

    # 2) 고정 경로 fallback (자동 탐색 실패 시)
    for dp in KNOWN_DATA_PATHS:
        if os.path.exists(dp):
            root = str(Path(dp).parent)
            for p in (root, dp):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print(f"[경로 설정] 고정 경로 사용 | 프로젝트 루트: {root}")
            print(f"[경로 설정] DATA 폴더: {dp}")
            return dp

    raise FileNotFoundError(
        f"DATA 폴더를 찾을 수 없습니다.\n"
        f"현재 위치: {current}\n"
        f"상위 폴더에 DATA 폴더가 있는지, 또는 KNOWN_DATA_PATHS의 경로가 맞는지 확인하세요."
    )

DATA_PATH = setup_universal_paths()

from DATA.hs_code_map_v2 import HS_CODE_MAP, validate_hs_code_map

_bad = validate_hs_code_map(HS_CODE_MAP)
if _bad:
    raise ValueError(f"hs_code_map_v2 에 형식이 잘못된 코드가 있습니다 (문자열/짝수 자릿수 아님): {_bad}")
# -----------------------------------------------------------------------------
# 로깅 설정
# -----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# -----------------------------------------------------------------------------
# 설정
# -----------------------------------------------------------------------------
# 서비스키: DATA/KEYS.py 의 KEYS 에서 아래 이름 순서로 찾음 (공공데이터포털 키)
SERVICE_KEY_NAMES = ("CUSTOMS", "ODPD")

START_YYYYMM = "202601"
END_YYYYMM   = "202612"
REGION_NAME  = "전국"

TARGET_CODES = None
#
# # YoY 계산용으로 시작월 12개월 전부터 함께 받기 (권장 True)
YOY_LOOKBACK = True
#
# # DB 업로드: None = 실행 중 물어봄 / True = 자동 업로드 / False = 업로드 안 함
AUTO_UPLOAD = None

# TARGET_CODES = ["030341", "030342", "030343", "030344", "030354", "030389",
#                "030487", "030633", "0207", "190490"]
# START_YYYYMM = "200701"
# AUTO_UPLOAD = True

RETRIES = 3
TIMEOUT = 60

MAX_WORKERS   = 8        # 8~10 추천
REQUEST_DELAY = 0.10     # 서버 안정성 위해 0.1~0.2 추천
CHUNK_SIZE    = 1000

# DB 접속: DATA/config.py 사용 (비밀번호를 코드에 적지 않음)
import config as investar_config
ENGINE = investar_config.get_engine(investar_config.get_db_info())

# 서비스키 로드
from urllib.parse import quote
try:
    from DATA.KEYS import KEYS
except Exception:
    KEYS = {}
_raw_key = next((KEYS.get(k) for k in SERVICE_KEY_NAMES if KEYS.get(k)), None) or os.environ.get("CUSTOMS_SERVICE_KEY")
if not _raw_key:
    raise RuntimeError(f"서비스키가 없습니다. DATA/KEYS.py 에 KEYS['CUSTOMS'] = '...' 를 추가하세요. "
                       f"(현재 KEYS 이름: {sorted(KEYS)})")
# URL에 직접 붙이므로 인코딩된 형태로 맞춤 (이미 %가 있으면 인코딩된 키로 간주)
SERVICE_KEY = _raw_key if "%" in _raw_key else quote(_raw_key, safe="")


def mask_key(msg) -> str:
    out = str(msg)
    for k in {SERVICE_KEY, _raw_key}:
        if k:
            out = out.replace(k, "***")
    return out

# Unique HS Code 리스트 추출
hs_code_list: list = list(HS_CODE_MAP.keys())

# -----------------------------------------------------------------------------
# 유틸
# -----------------------------------------------------------------------------
def norm_yymm(yymm: str) -> str:
    """YYYYMM 6자리 검증"""
    yymm = str(yymm).strip()
    if not re.fullmatch(r"\d{6}", yymm):
        raise ValueError(f"Invalid YYYYMM (must be 6 digits): {yymm}")
    mm = int(yymm[4:6])
    if not (1 <= mm <= 12):
        raise ValueError(f"Invalid month in YYYYMM: {yymm}")
    return yymm

def yymm_to_int(yymm: str) -> int:
    """YYYYMM -> month index (연*12 + (월-1))"""
    yymm = norm_yymm(yymm)
    return int(yymm[:4]) * 12 + (int(yymm[4:6]) - 1)

def int_to_yymm(m: int) -> str:
    """month index -> YYYYMM"""
    yyyy = m // 12
    mm = (m % 12) + 1
    return f"{yyyy:04d}{mm:02d}"

def split_into_12m_windows(start_yymm: str, end_yymm: str):
    """
    [start_yymm, end_yymm] (포함) 범위를
    API 제약(<=12개월)에 맞게 구간 리스트로 분할.
    """
    s = yymm_to_int(start_yymm)
    e = yymm_to_int(end_yymm)
    if e < s:
        raise ValueError("end_yymm must be >= start_yymm")

    windows = []
    cur = s
    while cur <= e:
        win_end = min(cur + 11, e)  # 최대 12개월
        windows.append((int_to_yymm(cur), int_to_yymm(win_end)))
        cur = win_end + 1
    return windows

def parse_items_to_df(response_text: str) -> pd.DataFrame:
    """
    API 응답(XML)을 안전하게 파싱해 items/item을 DF로 반환.
    정상 구조가 아니면 빈 DF.
    """
    try:
        parsed = xmltodict.parse(response_text)
    except Exception:
        return pd.DataFrame()

    if not isinstance(parsed, dict):
        return pd.DataFrame()

    resp = parsed.get("response")
    if not isinstance(resp, dict):
        return pd.DataFrame()

    body = resp.get("body")
    if not isinstance(body, dict):
        return pd.DataFrame()

    items = body.get("items")
    if not isinstance(items, dict):
        return pd.DataFrame()

    item = items.get("item")
    if item is None:
        return pd.DataFrame()

    if isinstance(item, dict):
        item = [item]
    if not isinstance(item, list):
        return pd.DataFrame()

    return pd.DataFrame(item)

def extract_api_header(response_text: str):
    """resultCode/resultMsg 추출 (없으면 None,None)"""
    try:
        parsed = xmltodict.parse(response_text)
        if not isinstance(parsed, dict):
            return None, None
        resp = parsed.get("response")
        if not isinstance(resp, dict):
            return None, None
        header = resp.get("header")
        if not isinstance(header, dict):
            return None, None
        return header.get("resultCode"), header.get("resultMsg")
    except Exception:
        return None, None

def make_session(max_workers: int) -> requests.Session:
    """연결 재사용 + pool 확장"""
    sess = requests.Session()
    adapter = HTTPAdapter(pool_connections=max_workers, pool_maxsize=max_workers)
    sess.mount("https://", adapter)
    sess.mount("http://", adapter)
    return sess

# -----------------------------------------------------------------------------
# 수집 함수 (단일 window)
# -----------------------------------------------------------------------------
def get_country_export_by_item(session: requests.Session,
                               service_key: str,
                               start_yymm: str,
                               end_yymm: str,
                               hs_code_raw) -> pd.DataFrame:
    """
    단일 window(start_yymm~end_yymm) 수집
    - 요청 HS는 6자리로 절단(안정성)
    - 원본 root_hs_code는 그대로 저장
    """
    start_yymm = norm_yymm(start_yymm)
    end_yymm   = norm_yymm(end_yymm)

    hs_code_raw = str(hs_code_raw).strip()
    hs_code_req = hs_code_raw[:6]

    url = (
        "https://apis.data.go.kr/1220000/Itemtrade/getItemtradeList"
        f"?serviceKey={service_key}"
        f"&strtYymm={start_yymm}"
        f"&endYymm={end_yymm}"
        f"&hsSgn={hs_code_req}"
    )

    for attempt in range(RETRIES):
        try:
            r = session.get(url, timeout=TIMEOUT)

            # 재시도 상태코드
            if r.status_code in (429, 502, 503, 504):
                if attempt < RETRIES - 1:
                    backoff = 2 ** attempt
                    logger.warning(f"Retryable {r.status_code} HS {hs_code_raw} ({start_yymm}-{end_yymm}) sleep {backoff}s")
                    time.sleep(backoff)
                    continue
                else:
                    logger.error(f"Retryable {r.status_code} but retries exhausted HS {hs_code_raw} ({start_yymm}-{end_yymm})")
                    return pd.DataFrame()

            r.raise_for_status()

            df = parse_items_to_df(r.text)
            if df.empty:
                code, msg = extract_api_header(r.text)
                if code or msg:
                    logger.warning(f"No/Err data HS {hs_code_raw} (req={hs_code_req}) {start_yymm}-{end_yymm} code={code} msg={msg}")
                else:
                    logger.warning(f"No data HS {hs_code_raw} (req={hs_code_req}) {start_yymm}-{end_yymm} head={r.text[:200]}")
                return pd.DataFrame()

            df["root_hs_code"] = hs_code_raw
            df["hsSgn_req"] = hs_code_req
            df["req_start_yymm"] = start_yymm
            df["req_end_yymm"] = end_yymm

            if REQUEST_DELAY > 0:
                time.sleep(REQUEST_DELAY)

            return df

        except requests.exceptions.RequestException as e:
            if attempt < RETRIES - 1:
                backoff = 2 ** attempt
                logger.warning(f"Attempt {attempt+1}/{RETRIES} failed HS {hs_code_raw} ({start_yymm}-{end_yymm}): {mask_key(e)} | sleep {backoff}s")
                time.sleep(backoff)
                continue
            logger.error(f"All retries failed HS {hs_code_raw} ({start_yymm}-{end_yymm}): {mask_key(e)}")
            return pd.DataFrame()

        except Exception as e:
            logger.error(f"Unexpected error HS {hs_code_raw} ({start_yymm}-{end_yymm}): {mask_key(e)}")
            return pd.DataFrame()

# -----------------------------------------------------------------------------
# 수집 함수 (HS당 여러 window 자동 분할)
# -----------------------------------------------------------------------------
def fetch_hs_multiwindow(session: requests.Session,
                         service_key: str,
                         start_yymm: str,
                         end_yymm: str,
                         hs_code_raw) -> pd.DataFrame:
    """
    HS 하나에 대해 전체 기간을 12개월 window로 쪼개서 수집 후 concat
    """
    windows = split_into_12m_windows(start_yymm, end_yymm)
    dfs = []

    for (s, e) in windows:
        df = get_country_export_by_item(session, service_key, s, e, hs_code_raw)
        if df is not None and not df.empty:
            dfs.append(df)

    if dfs:
        return pd.concat(dfs, ignore_index=True)
    return pd.DataFrame()

# -----------------------------------------------------------------------------
# 전처리/집계
# -----------------------------------------------------------------------------
def process_and_aggregate(df: pd.DataFrame, region_name: str):
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    # 총계 제거(있을 때만)
    if "year" in df.columns:
        df = df[df["year"] != "총계"].copy()

    # 날짜 처리
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if "year" in df.columns:
            df["new_date"] = pd.to_datetime(df["year"].astype(str).str.replace(".", "-"), errors="coerce") + MonthEnd(0)
        else:
            df["new_date"] = pd.NaT

    df = df.dropna(subset=["new_date"]).copy()
    df = df.set_index("new_date")

    df["new_year"] = df.index.year
    df["new_quarter"] = df.index.quarter
    df["new_month"] = df.index.month

    numeric_cols = ["balPayments", "expDlr", "expWgt", "impDlr", "impWgt"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
        else:
            df[col] = 0.0

    base_cols = ["hsCode", "statKor", "root_hs_code", "hsSgn_req"]
    time_cols = ["new_year", "new_quarter", "new_month"]
    keep_cols = [c for c in (base_cols + time_cols + numeric_cols) if c in df.columns]

    clean_df = df[keep_cols].copy()
    clean_df["region"] = region_name

    agg_dict = {"balPayments": "sum", "expDlr": "sum", "impDlr": "sum", "expWgt": "sum", "impWgt": "sum"}

    by_m = (clean_df.groupby(["root_hs_code", "new_year", "new_quarter", "new_month"])
            .agg(agg_dict)
            .reset_index())
    by_m["region"] = region_name

    by_q = (clean_df.groupby(["root_hs_code", "new_year", "new_quarter"])
            .agg(agg_dict)
            .reset_index())
    by_q["region"] = region_name

    return by_q, by_m

def add_yoy_growth(df: pd.DataFrame, steps: int) -> pd.DataFrame:
    if df.empty:
        return df

    out = df.copy()

    if steps == 12:
        out["date"] = pd.to_datetime(out["new_year"].astype(str) + "-" +
                                     out["new_month"].astype(str).str.zfill(2) + "-01") + MonthEnd(0)
    elif steps == 4:
        end_month_map = {1: "03", 2: "06", 3: "09", 4: "12"}
        end_month = out["new_quarter"].map(end_month_map)
        out["date"] = pd.to_datetime(out["new_year"].astype(str) + "-" + end_month + "-01") + MonthEnd(0)
    else:
        raise ValueError("steps는 12(월) 또는 4(분기)만 지원합니다.")

    out = out.sort_values(["root_hs_code", "date"])
    g = out.groupby("root_hs_code")
    out["expDlr_yoy"] = g["expDlr"].transform(lambda x: x.pct_change(periods=steps))
    out["impDlr_yoy"] = g["impDlr"].transform(lambda x: x.pct_change(periods=steps))

    return out

def reshape_to_long(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    id_vars = ["date", "root_hs_code"]
    value_vars = ["balPayments", "expDlr", "impDlr", "expWgt", "impWgt", "expDlr_yoy", "impDlr_yoy"]
    value_vars = [c for c in value_vars if c in df.columns]

    long_df = df.melt(id_vars=id_vars, value_vars=value_vars,
                      var_name="indicator", value_name="value")
    long_df = long_df.dropna(subset=["value"])
    return long_df

# -----------------------------------------------------------------------------
# DB 업로드
# -----------------------------------------------------------------------------
def upload_to_db_optimized(df_long: pd.DataFrame, table_name: str):
    """INSERT ... ON DUPLICATE KEY UPDATE 로 저장 (새 키는 추가, 기존 키는 값 갱신)."""
    if df_long.empty:
        logger.info(f"업로드할 데이터가 비어있습니다 ({table_name}).")
        return

    df_long = df_long.copy()
    df_long["date"] = pd.to_datetime(df_long["date"]).dt.date
    df_long["root_hs_code"] = df_long["root_hs_code"].astype(str)
    df_long = df_long.replace([np.inf, -np.inf], np.nan).dropna(subset=["value"])
    df_long["value"] = df_long["value"].astype(float)
    recs = df_long[["date", "root_hs_code", "indicator", "value"]].to_dict("records")

    create_sql = text(f"""
    CREATE TABLE IF NOT EXISTS `{table_name}` (
        `date` DATE,
        `root_hs_code` VARCHAR(20),
        `indicator` VARCHAR(50),
        `value` DOUBLE,
        PRIMARY KEY (`date`, `root_hs_code`, `indicator`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """)
    upsert_sql = text(f"""
    INSERT INTO `{table_name}` (`date`, `root_hs_code`, `indicator`, `value`)
    VALUES (:date, :root_hs_code, :indicator, :value)
    ON DUPLICATE KEY UPDATE `value` = VALUES(`value`)
    """)

    logger.info(f"업로드 대상({table_name}): {len(recs):,}건 (새 키 추가 + 기존 키 갱신)")
    with ENGINE.begin() as conn:
        conn.execute(create_sql)
        for i in tqdm(range(0, len(recs), CHUNK_SIZE), desc=f"DB 업로드({table_name})"):
            conn.execute(upsert_sql, recs[i:i + CHUNK_SIZE])
    logger.info(f"총 {len(recs):,}건 저장 완료 ({table_name})")

# =============================================================================
# 메인
# =============================================================================
def main():
    t0 = time.time()

    # YYYYMM 검증
    norm_yymm(START_YYYYMM)
    norm_yymm(END_YYYYMM)

    fetch_start = START_YYYYMM
    if YOY_LOOKBACK:
        fetch_start = int_to_yymm(yymm_to_int(START_YYYYMM) - 12)   # YoY 계산용 12개월 앞부터
    windows = split_into_12m_windows(fetch_start, END_YYYYMM)
    logger.info(f"전체 수집기간: {fetch_start}~{END_YYYYMM} (저장 기준 {START_YYYYMM}~) | window 개수: {len(windows)}")

    # HS 코드 로드
    # hs_data = fetch_table_data(DB_INFO, "target_hs_code")
    # HS_CODES = hs_data["hs_code"].dropna().astype(str).unique().tolist()

    HS_CODES: list = [str(c).strip() for c in TARGET_CODES] if TARGET_CODES else list(HS_CODE_MAP.keys())
    if TARGET_CODES:
        logger.info(f"TARGET_CODES 지정: {HS_CODES}")


    logger.info("무역 데이터 수집 시작")
    logger.info(f"HS 코드 수: {len(HS_CODES)}")
    logger.info(f"동시 워커: {MAX_WORKERS}")

    session = make_session(MAX_WORKERS)

    all_dfs = []
    errors = []
    empty_count = 0

    try:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_map = {
                executor.submit(fetch_hs_multiwindow, session, SERVICE_KEY, fetch_start, END_YYYYMM, hs): hs
                for hs in HS_CODES
            }

            for future in tqdm(as_completed(future_map), total=len(future_map), desc="API 요청"):
                hs = future_map[future]
                try:
                    df = future.result(timeout=300)
                    if df is not None and not df.empty:
                        all_dfs.append(df)
                    else:
                        empty_count += 1
                except Exception as e:
                    logger.error(f"HS {hs} 처리 실패: {mask_key(e)}")
                    errors.append(hs)

    finally:
        session.close()

    logger.info(f"수집 종료. 소요: {time.time()-t0:.2f}초 | 실패 HS: {len(errors)} | 빈결과 HS: {empty_count}")

    if not all_dfs:
        logger.error("수집된 데이터가 없습니다.")
        return None

    final_df = pd.concat(all_dfs, ignore_index=True)
    logger.info(f"총 수집 레코드: {len(final_df):,}건")

    processed_q, processed_m = process_and_aggregate(final_df, REGION_NAME)
    logger.info(f"월별 집계: {len(processed_m):,}건 | 분기별 집계: {len(processed_q):,}건")

    if processed_m.empty:
        logger.warning("처리할 월별 데이터가 없습니다.")
        return None

    monthly_with_yoy = add_yoy_growth(processed_m, steps=12)
    quarterly_with_yoy = add_yoy_growth(processed_q, steps=4)

    monthly_long = reshape_to_long(monthly_with_yoy)
    quarterly_long = reshape_to_long(quarterly_with_yoy)

    logger.info(f"월별 long: {len(monthly_long):,}건 | 분기별 long: {len(quarterly_long):,}건")

    if AUTO_UPLOAD is None:
        do_upload = input("데이터베이스에 업로드하시겠습니까? (y/n): ").strip().lower() == "y"
    else:
        do_upload = bool(AUTO_UPLOAD)
    if do_upload:
        upload_to_db_optimized(monthly_long, "korea_monthly_trade_data")
        upload_to_db_optimized(quarterly_long, "korea_quarterly_trade_data")

    logger.info("전체 파이프라인 완료")

    return {
        "final_raw": final_df,
        "monthly": monthly_with_yoy,
        "quarterly": quarterly_with_yoy,
        "monthly_long": monthly_long,
        "quarterly_long": quarterly_long,
        "errors": errors,
        "empty_hs_count": empty_count,
    }

# 실행
out = main()

# 결과 변수 편의 할당
if out:
    final_raw = out["final_raw"]
    monthly_trade_data = out["monthly"]
    quarterly_trade_data = out["quarterly"]
    monthly_long_data = out["monthly_long"]
    quarterly_long_data = out["quarterly_long"]
    error_list = out["errors"]

    print("\n사용 가능한 변수들:")
    print("- final_raw")
    print("- monthly_trade_data")
    print("- quarterly_trade_data")
    print("- monthly_long_data")
    print("- quarterly_long_data")
    print("- error_list")

# -----------------------------------------------------------------------------
# 시각화 함수(유지)
# -----------------------------------------------------------------------------
def plot_column_by_hscode(df, hs_code, col_name, start_date=None, end_date=None):
    import matplotlib.pyplot as plt

    if df is None or df.empty:
        print("DataFrame이 비어 있습니다.")
        return
    if "date" not in df.columns:
        print("'date' 컬럼이 없습니다. add_yoy_growth()를 먼저 실행하세요.")
        return

    hs_code = str(hs_code)
    target_df = df[df["root_hs_code"].astype(str) == hs_code].sort_values("date")

    if target_df.empty:
        print(f" root_hs_code {hs_code}에 해당하는 데이터가 없습니다.")
        return

    if col_name not in target_df.columns:
        print(f" '{col_name}' 컬럼이 DataFrame에 없습니다.")
        return

    if start_date:
        target_df = target_df[target_df["date"] >= pd.to_datetime(start_date)]
    if end_date:
        target_df = target_df[target_df["date"] <= pd.to_datetime(end_date)]

    if target_df.empty:
        print("지정한 날짜 범위에 데이터가 없습니다.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(target_df["date"], target_df[col_name], marker="o", label=col_name)

    last_x = target_df["date"].iloc[-1]
    last_y = target_df[col_name].iloc[-1]

    if "yoy" in col_name:
        plt.text(last_x, last_y, f"{last_y * 100:,.2f}%", fontsize=12, ha="left", va="bottom", color="red")
    else:
        plt.text(last_x, last_y, f"{last_y:,.0f}", fontsize=12, ha="left", va="bottom", color="red")

    plt.title(f"{col_name} 추이 (root_hs_code: {hs_code})")
    plt.xlabel("Date")
    plt.ylabel(col_name)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

2026-10-03 19:53:42,186 - INFO - 전체 수집기간: 200601~202612 (저장 기준 200701~) | window 개수: 21
2026-10-03 19:53:42,187 - INFO - TARGET_CODES 지정: ['030341', '030342', '030343', '030344', '030354', '030389', '030487', '030633', '0207', '190490']
2026-10-03 19:53:42,187 - INFO - 무역 데이터 수집 시작
2026-10-03 19:53:42,188 - INFO - HS 코드 수: 10
2026-10-03 19:53:42,188 - INFO - 동시 워커: 8


[경로 설정] 자동 탐색 성공 | 프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[경로 설정] DATA 폴더: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA


API 요청:   0%|          | 0/10 [00:00<?, ?it/s]2026-10-03 19:53:42,403 - WARNING - No/Err data HS 030487 (req=030487) 200601-200612 code=00 msg=정상서비스.
2026-10-03 19:53:42,406 - WARNING - No/Err data HS 030389 (req=030389) 200601-200612 code=00 msg=정상서비스.
2026-10-03 19:53:42,417 - WARNING - No/Err data HS 030633 (req=030633) 200601-200612 code=00 msg=정상서비스.
2026-10-03 19:53:42,565 - WARNING - No/Err data HS 030487 (req=030487) 200701-200712 code=00 msg=정상서비스.
2026-10-03 19:53:42,566 - WARNING - No/Err data HS 030633 (req=030633) 200701-200712 code=00 msg=정상서비스.
2026-10-03 19:53:42,572 - WARNING - No/Err data HS 030389 (req=030389) 200701-200712 code=00 msg=정상서비스.
2026-10-03 19:53:42,703 - WARNING - No/Err data HS 030487 (req=030487) 200801-200812 code=00 msg=정상서비스.
2026-10-03 19:53:42,716 - WARNING - No/Err data HS 030633 (req=030633) 200801-200812 code=00 msg=정상서비스.
2026-10-03 19:53:42,717 - WARNING - No/Err data HS 030389 (req=030389) 200801-200812 code=00 msg=정상서비스.
2026-10-03 19:53:4


사용 가능한 변수들:
- final_raw
- monthly_trade_data
- quarterly_trade_data
- monthly_long_data
- quarterly_long_data
- error_list


In [3]:
monthly_df = out['monthly']
monthly_df
monthly_df[monthly_df['root_hs_code'] == '854232'].tail(13)

,root_hs_code,new_year,new_quarter,new_month,balPayments,expDlr,impDlr,expWgt,impWgt,region,date,expDlr_yoy,impDlr_yoy


In [4]:
# date 인덱스 생성
monthly_df['date'] = pd.to_datetime(
    monthly_df['new_year'].astype(str) + '-' + monthly_df['new_month'].astype(str).str.zfill(2)
)

# pivot table 생성
pivot_df = monthly_df.pivot_table(
    index='date',
    columns='root_hs_code',
    values='expDlr',
    aggfunc='sum'
)

pivot_df.index = pd.DatetimeIndex(pivot_df.index).to_period('M')

print(pivot_df)

root_hs_code       0207     030341     030342      030343     030344  \
date                                                                   
2006-01        524975.0   250198.0  2969303.0   2537809.0  2902647.0   
2006-02        469415.0   258867.0  4654907.0      2099.0  6150186.0   
2006-03        257778.0  1090770.0  3349235.0   4728086.0  7990670.0   
2006-04        228814.0   223344.0  5557376.0   4227692.0  6997467.0   
2006-05        317613.0   376205.0  2971021.0   4806492.0  3553483.0   
...                 ...        ...        ...         ...        ...   
2026-04       5243114.0   726074.0  5955960.0  21608352.0  3695398.0   
2026-05       3929517.0   680576.0  4214018.0  16649865.0  4555163.0   
2026-06       3963221.0   498148.0  5604700.0  13167232.0  2907825.0   
2026-07       5851999.0   416523.0  7337526.0  11462398.0  3413307.0   
2026-08       4565948.0    96979.0  1302301.0   2177291.0   976600.0   

root_hs_code      030354      030389      030487     030633    

In [4]:
# pivot_df.to_csv(r'C:\Users\82108\OneDrive\INVESTMENT\Data_Asnal\Export_data\export_202602.csv')

In [5]:
pivot_df[['854232']].tail(15)

root_hs_code,854232
date,
2025-06,8.230251e+09
2025-07,7.560781e+09
2025-08,8.504645e+09
2025-09,9.345975e+09
2025-10,8.886616e+09
2025-11,1.016366e+10
2025-12,1.263691e+10
2026-01,1.211196e+10
2026-02,1.581075e+10
